In [ ]:
# --- import libarary ---
import pandas as pd
import networkx as nx
import numpy as np

# Load edge list and build graph
ppi_df = pd.read_csv('gene_relationship.csv')
# Load drug-target mappings
drug_df = pd.read_csv("drug_targets.csv")

# --- load data ---
gene_list = drug_df.symbol.unique()
ppi_gene1 = ppi_df['symbol_1'].unique()
ppi_gene2 = ppi_df['symbol_2'].unique()
ppi_gene = np.unique(np.concatenate([ppi_gene1, ppi_gene2]))

In [ ]:
# --- Data Preprocessing ---
genes_in_list = set(gene_list)
genes_in_ppi = set(ppi_gene)

# Find genes present in gene_list but missing from the PPI network
missing_genes = genes_in_list - genes_in_ppi

# View the result
print(f"{len(missing_genes)} genes are not in the PPI network.")
print(missing_genes)

In [ ]:
ppi_gene_set = set(ppi_gene)
drug_df_filtered = drug_df[drug_df['symbol'].isin(ppi_gene_set)].copy()

In [ ]:
G = nx.from_pandas_edgelist(ppi_df, source="symbol_1", target="symbol_2")

In [ ]:
C = drug_df_filtered[drug_df_filtered["drug"] == "Carfilzomib"]["symbol"].tolist()
A = drug_df_filtered[drug_df_filtered["drug"] == "Adalimumab"]["symbol"].tolist()
S = drug_df_filtered[drug_df_filtered["drug"] == "Secukinumab"]["symbol"].tolist()

In [ ]:
import networkx as nx
import numpy as np

def mean_pairwise_distance(G, X, Y):
    """
    Compute the mean shortest path length between all pairs in X and Y.
    """
    distances = []
    for x in X:
        for y in Y:
            if x == y:
                distances.append(0)
            elif G.has_node(x) and G.has_node(y):
                try:
                    dist = nx.shortest_path_length(G, x, y)
                    distances.append(dist)
                except nx.NetworkXNoPath:
                    continue 
    return np.mean(distances) if distances else np.inf

def s_score(G, A_targets, B_targets):
    """
    Compute the s_score between two drug target modules A and B.

    s_AB = mean(d_AB) - (mean(d_AA) + mean(d_BB)) / 2
    """
    dAB = mean_pairwise_distance(G, A_targets, B_targets)
    dAA = mean_pairwise_distance(G, A_targets, A_targets)
    dBB = mean_pairwise_distance(G, B_targets, B_targets)
    
    # If A and B are completely disconnected, return infinity
    if np.isinf(dAB): # np.isinf: checks whether the value is infinity
        return np.inf
     # If both modules are internally disconnected, s_score is undefined
    if np.isinf(dAA) and np.isinf(dBB):
        return np.inf  # all disconnected
    return dAB - ((dAA + dBB) / 2)


In [ ]:
s_score(G, C, A)

In [ ]:
s_score(G, C, S)

In [ ]:
s_score(G, S, A)